# Herramienta Interactiva de Visualización y Análisis Espaciotemporal de la Sismicidad en Chile
### Cuaderno de Trabajo y Experimentación: Caso de Estudio Terremoto de Illapel 2015 ($M_w\ 8.3$)

**Asignatura:** EIN092B Visualización de Datos  
**Profesor:** Jorge Portilla  
**Autores:** Fabian Acuña, Fernando Gutiérrez  

---

## Propósito y Estructura General del Proyecto
Este cuaderno sirve como base experimental y metodológica para la **Segunda Entrega (Informe Final en Formato Paper IEEE)**.

El objetivo es estudiar la secuencia sísmica asociada al terremoto de Illapel del 16 de septiembre de 2015 ($M_w\ 8.3$), integrando técnicas de análisis espaciotemporal y agrupamiento no supervisado (DBSCAN) para superar las limitaciones de los catálogos tradicionales en 2D y tablas estáticas.

A continuación se detallan las **7 etapas lógicas** que desarrollaremos juntos paso a paso.

---
## Etapa 1: Adquisición Automática y Reproducible de Datos

### ¿Qué haremos en esta etapa?
- Definir los parámetros de consulta a la API pública oficial del **Servicio Geológico de Estados Unidos (USGS)**.
- Delimitar la ventana temporal: desde el **15 de septiembre de 2015** hasta el **15 de octubre de 2015** (1 mes que captura el evento principal y la secuencia de réplicas críticas).
- Delimitar la región geográfica: Zona centro-norte de Chile (Latitud [-35.0, -28.0], Longitud [-74.0, -69.0]).
- Definir un umbral de magnitud mínima: **$M \ge 3.0$** para concentrar el análisis en eventos perceptibles o significativos y reducir el ruido instrumental.
- Implementar la lógica para que el código **descargue automáticamente** el archivo CSV si no se encuentra en el entorno, garantizando que cualquier persona que abra este cuaderno (como el profesor) pueda ejecutarlo sin pasos manuales previos.

### Variables esperadas de la API:
- `time`: Fecha y hora UTC del evento.
- `latitude`, `longitude`: Coordenadas del epicentro.
- `depth`: Profundidad del hipocentro en kilómetros (variable fundamental que las plataformas tradicionales no visualizan adecuadamente en 2D).
- `mag`: Magnitud del sismo.
- `place`: Descripción geográfica de referencia.

In [ ]:
# [Etapa 1] Código de adquisición y descarga automática
# (Aquí programaremos juntos la consulta a la API del USGS)


---
## Etapa 2: Exploración, Limpieza y Estructuración de Datos

### ¿Qué haremos en esta etapa?
- **Control de integridad:** Identificar y tratar valores nulos, registros duplicados o inconsistencias en los campos numéricos.
- **Normalización temporal:** Convertir las cadenas de texto del campo `time` a objetos `datetime` de Pandas con zona horaria UTC explícita, ordenando cronológicamente el catálogo.
- **Detección automática del sismo principal (*Mainshock*):** Localizar mediante código el evento de mayor magnitud ($M_w\ 8.3$) para usarlo como punto de referencia espacial y temporal.
- **Resumen estadístico inicial:** Conocer la cantidad total de sismos, rango de profundidades y distribución de magnitudes antes de aplicar modelos.

In [ ]:
# [Etapa 2] Código de exploración, limpieza y detección del evento principal
# (Aquí programaremos juntos el filtrado y estructuración del DataFrame)


---
## Etapa 3: Justificación y Selección Objetiva de Hiperparámetros (k-Distance Graph)

### ¿Qué haremos en esta etapa?
> **Importante para la evaluación:** En la primera entrega, el profesor indicó la necesidad de sustentar la selección de modelos y parámetros con criterios objetivos rigurosos.

- El algoritmo de agrupamiento **DBSCAN** requiere dos parámetros críticos:
  1. $\varepsilon$ (*epsilon*): Radio de vecindad para considerar dos eventos como vecinos.
  2. $MinPts$ (*min_samples*): Número mínimo de sismos dentro del radio $\varepsilon$ para formar una zona densa (núcleo).
- Para el espacio tridimensional ($X, Y, Z$ = Latitud, Longitud, Profundidad), normalizaremos previamente las variables para que los grados y los kilómetros de profundidad tengan escalas comparables.
- Basándonos en la literatura técnica ($MinPts \ge 2 \times \text{dimensiones}$), fijaremos $MinPts = k$.
- Calcularemos la distancia de cada sismo a su $k$-ésimo vecino más cercano y graficaremos la curva ordenada de distancias (**k-distance graph**).
- Identificaremos el punto de máxima curvatura (**método del codo**) para fijar $\varepsilon$ de manera matemáticamente fundamentada, no arbitraria.
- Este gráfico será exportado como la **Figura 2** de nuestro paper.

In [ ]:
# [Etapa 3] Código del k-distance graph y método del codo
# (Aquí programaremos el cálculo de vecinos más cercanos y la exportación de la Figura 2)


---
## Etapa 4: Modelado No Supervisado con DBSCAN (Clustering Espacial 3D)

### ¿Qué haremos en esta etapa?
- Ejecutar DBSCAN sobre el espacio normalizado 3D con los parámetros $\varepsilon$ y $MinPts$ justificados en la etapa anterior.
- Clasificar los eventos en dos categorías sismológicas clave:
  1. **Clusters densos (núcleos de réplicas):** Sismos estrechamente vinculados a la zona de ruptura de la falla geológica.
  2. **Ruido / Outliers (etiqueta -1):** Sismicidad de fondo dispersa o eventos regionales no asociados directamente al plano de ruptura del terremoto de Illapel.
- Cuantificar cuántos eventos pertenecen a la secuencia activa y cuántos corresponden a sismicidad aislada.

In [ ]:
# [Etapa 4] Código de entrenamiento DBSCAN y asignación de etiquetas
# (Aquí ejecutaremos el clustering y analizaremos la distribución de clusters)


---
## Etapa 5: Generación de las Figuras Oficiales para el Paper (Máximo 4)

### Requisitos estrictos de la rúbrica y la plantilla guía:
- Máximo **4 figuras o tablas en total** en todo el artículo.
- Resolución técnica alta (`dpi=300`).
- Pie de figura (*caption*) descriptivo y autosuficiente (sin títulos superiores redundantes).
- Cada figura debe responder directamente a uno de los objetivos planteados en la investigación.

### Las 4 figuras planificadas:
1. **Figura 1:** *Mapa 2D de sismicidad epicentral.* Muestra la distribución espacial tradicional en el plano de Chile, destacando el epicentro del sismo principal ($M_w\ 8.3$) y el tamaño según magnitud.
2. **Figura 2:** *Gráfico de distancias k (k-distance graph).* Evidencia metodológica del ajuste objetivo del parámetro $\varepsilon$ para DBSCAN.
3. **Figura 3:** *Perfil tridimensional (3D) de hipocentros.* Demuestra el plano de inclinación de la Placa de Nazca subduciendo bajo la Placa Sudamericana, resolviendo la principal limitación de los mapas 2D.
4. **Figura 4:** *Segmentación de réplicas con DBSCAN.* Mapa que distingue visualmente los grupos densos de réplicas frente a la sismicidad dispersa (ruido).

In [ ]:
# [Etapa 5] Código de generación y guardado en alta resolución (300 DPI) de las 4 figuras
# (Aquí configuraremos los gráficos con formato estético para publicación IEEE)


---
## Etapa 6: Métricas Cuantitativas para la Sección de Resultados y Discusión

### ¿Qué haremos en esta etapa?
- Extraer una tabla resumen con datos numéricos consolidados para redactar el análisis objetivo en el informe:
  * Número total de eventos procesados ($M \ge 3.0$).
  * Magnitud máxima ($8.3$) y magnitud promedio de réplicas.
  * Profundidad media y máxima registrada (rango del plano sismogénico).
  * Cantidad y porcentaje de eventos agrupados por DBSCAN en la zona de ruptura.
  * Cantidad y porcentaje de sismos clasificados como sismicidad de fondo / ruido.
- Estos valores serán citados textualmente en el informe en la sección *IV. RESULTADOS Y DISCUSIÓN*.

In [ ]:
# [Etapa 6] Código de extracción y consolidación de métricas cuantitativas
# (Aquí construiremos la tabla de resumen estadístico)


---
## Etapa 7: Documentación y Arquitectura del Dashboard Interactivo (pp.py)

### Propósito del Módulo de Producción
Una vez completado y validado el análisis científico en este cuaderno, los modelos de agrupamiento y las técnicas de visualización se trasladan a una aplicación web interactiva independiente alojada en el archivo pp.py del repositorio.

Esta aplicación permite a usuarios no técnicos, tomadores de decisiones y evaluadores explorar dinámicamente los datos sin requerir conocimientos de programación.

### Estructura y Componentes de pp.py:
1. **Módulo de Adquisición y Caché:**
   - Utiliza @st.cache_data para almacenar en memoria el catálogo preprocesado y los resultados del clustering DBSCAN, garantizando tiempos de respuesta instantáneos en la interfaz.
   - Descarga autónoma desde la API del USGS en caso de que el archivo illapel_2015_usgs.csv no se encuentre localmente.

2. **Panel de Control Lateral (Filtros Dinámicos):**
   - **Selector de rango de magnitud:** Permite aislar eventos de mayor energía ( \ge 6.0$) o analizar la totalidad de las réplicas.
   - **Selector de ventana temporal:** Filtro dinámico por calendario para observar la evolución temporal de la sismicidad.
   - **Filtro de clusters sismológicos:** Permite contrastar la secuencia de réplicas en la falla activa contra los sismos de fondo (ruido).

3. **Panel de Métricas Clave (KPIs):**
   - Cuatro tarjetas superiores que se recalculan en tiempo real: total de eventos visibles, magnitud máxima, profundidad promedio y porcentaje de sismos en la falla activa.

4. **Pestañas de Visualización:**
   - **Pestaña 1 (Mapa Geoespacial 2D):** Implementa un mapa interactivo con OpenStreetMap (scatter_map), respondiendo a la retroalimentación del profesor en la Entrega 1. Permite hacer zoom sobre las localidades afectadas (Illapel, Salamanca, Los Vilos, Coquimbo).
   - **Pestaña 2 (Perfil 3D de Subducción):** Visor tridimensional (scatter_3d) con rotación continua en 360° y zoom para examinar la profundización de los hipocentros hacia el continente.
   - **Pestaña 3 (Evolución Temporal):** Diagrama de dispersión temporal que refleja la ley de decaimiento de réplicas de Omori.

### Instrucciones de Ejecución Local:
Para poner en marcha la aplicación web, ejecutar en la terminal del proyecto:
`ash
python -m streamlit run app.py
`